# Phase 6.4E — Specification–Template Alignment
Zero API calls. Reuse the frozen Phase 6.4D validation and Phase 6.3.3.1 library context. Compare the original D recommendation, automatic rule-only alignment and development-reviewed alignment. No gold labels enter alignment. Only positive structural evidence may revise a recommendation; missing fields do not establish incompatibility. No preset confirmation or DOCX generation.

In [ ]:
from pathlib import Path
import sys, subprocess, importlib, shutil, json
REPO_URL='https://github.com/wohaofanaweishenme-cell/tfl-generation-agent.git'
BRANCH='simplified-rag-v3'
ROOT=Path('/content/tfl-simplified-rag-v3')
if not ROOT.exists():
    subprocess.run(['git','clone','--depth','1','--branch',BRANCH,REPO_URL,str(ROOT)],check=True)
else:
    subprocess.run(['git','-C',str(ROOT),'fetch','origin',BRANCH],check=True)
    subprocess.run(['git','-C',str(ROOT),'checkout','--detach','FETCH_HEAD'],check=True)
subprocess.run([sys.executable,'-m','pip','install','-q','-e',f'{ROOT}[retrieval,test]'],check=True)
SRC=ROOT/'src'
sys.path.insert(0,str(SRC))
for name in list(sys.modules):
    if name=='tfl_agent' or name.startswith('tfl_agent.'):
        del sys.modules[name]
for p in SRC.joinpath('tfl_agent').rglob('__pycache__'):
    shutil.rmtree(p)
importlib.invalidate_caches()
import tfl_agent
assert Path(tfl_agent.__file__).resolve().is_relative_to(SRC.resolve())
print('Commit:',subprocess.check_output(['git','-C',str(ROOT),'rev-parse','HEAD'],text=True).strip())
subprocess.run([sys.executable,'-m','pytest','-q',str(ROOT/'tests/test_phase6_template_alignment.py')],check=True)

In [ ]:
from google.colab import drive
from datetime import datetime, timezone
from tfl_agent.agent.phase6_template_alignment import (
    VALIDATION_NAME,VALIDATION_SHA,PREFLIGHT_NAME,PREFLIGHT_SHA,frozen,run)
drive.mount('/content/drive')
BASE=Path('/content/drive/MyDrive/Colab Notebooks/agent/midterm demo')
def frozen_file(folder,name,expected):
    root=BASE/folder
    direct=root/name
    if direct.is_file():
        with frozen(direct,name,expected): pass
        return direct
    matches=[]
    for p in root.glob('*.zip'):
        try:
            with frozen(p,name,expected): pass
        except (ValueError,OSError,KeyError): continue
        matches.append(p)
    if len(matches)!=1:
        raise FileNotFoundError(f'Expected {direct} with SHA256 {expected}; matches: {matches}')
    return matches[0]
VALIDATION=frozen_file('Phase64D_Validation_Outputs',VALIDATION_NAME,VALIDATION_SHA)
PREFLIGHT=frozen_file('Phase6331_Simplified_Replay_Outputs',PREFLIGHT_NAME,PREFLIGHT_SHA)
print('Verified validation:',VALIDATION)
print('Verified library context:',PREFLIGHT)
RUN_ID=datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%S%fZ')
OUT=Path('/content/Phase64E_Alignment')/RUN_ID
DEST=BASE/'Phase64E_Template_Alignment_Outputs'
DEST.mkdir(parents=True,exist_ok=True)

In [ ]:
report,decisions=run(VALIDATION,PREFLIGHT,OUT)
assert report['new_llm_calls']==0 and len(decisions)==18
assert report['matrix_rows']==108
assert not report['generation_authorized']
assert all(d['confirmed_template'] is None and d['confirmed_preset'] is None for d in decisions)
print(json.dumps(report,indent=2))
import pandas as pd
from IPython.display import display
comparison=pd.read_csv(OUT/'Paired_Comparison.csv')
display(pd.DataFrame(report['metrics']).T)
display(comparison[['study_id','entry_id','arm','expected','recommended_template','decision']])
gaps=pd.read_csv(OUT/'Required_Field_Gaps.csv')
display(gaps.groupby(['arm','status']).size().unstack(fill_value=0))
saved=Path(shutil.make_archive(str(DEST/RUN_ID),'zip',root_dir=OUT))
print('Saved:',saved)
print('Files:',sorted(p.name for p in OUT.iterdir()))

## Interpretation
Original matching is the Phase 6.3.3.1 replay of D_applicability, not the title-only baseline. Both alignment arms use the same conservative rules. Reviewed alignment is a development-reference arm, not a fully automated workflow. A provisional revision is a recommendation, not confirmation. No evidence is manufactured for missing fields. The required-field audit lists actual template paths but does not bind them to study values. Rule coverage is limited to selected structural signatures and heuristic cues; all nine candidates and not_for descriptions remain visible for review. This development cohort is not held-out.